# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamemad975/FlyRank_ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For CTR early-warning model, we compare Random Forest and Gradient Boosting (LightGBM/XGBoost) with the existing Logistic Regression baseline.

Why?
CTR and search-position changes can have non-linear relationships and interactions. For example, a page with many impressions may become a stronger warning signal when its search position also gets worse. Tree-based models can capture these patterns without needing to manually create interaction features.

Overfitting control:
To reduce overfitting, we limit the tree depth and set a minimum number of samples per leaf. This helps prevent the model from learning noise or temporary patterns in the data.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/mariamemad975/FlyRank_ML.git
%cd FlyRank_ML

Cloning into 'FlyRank_ML'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 174 (delta 77), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 1.92 MiB | 8.18 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/FlyRank_ML/FlyRank_ML


In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
import lightgbm as lgb

In [22]:
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv") or file.endswith(".parquet"):
            print(os.path.join(root, file))

./data/raw/content_refresh_anonymized.csv
./outputs/refresh_queue_sample.csv


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Will use GroupKFold with 5 folds, grouping the data by content_hash_id.

Why?
The same content can appear in multiple records, so its data may be very similar across different snapshots.

Avoiding data leakage:
A random split could put the same page in both the training and validation sets, making the model look better than it really is. By grouping by content_hash_id, all records from the same page stay in the same fold.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [24]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [25]:
TARGET = 'is_decay_risk' if 'is_decay_risk' in df.columns else df.columns[-1]
GROUP = 'content_hash_id' if 'content_hash_id' in df.columns else df.columns[0]
df = df.dropna(subset=[TARGET])

In [26]:
if df[TARGET].nunique() > 2:
    df[TARGET] = (df[TARGET] > df[TARGET].median()).astype(int)
else:
    df[TARGET] = df[TARGET].astype(int)

In [27]:
features = df.select_dtypes(include=np.number).columns.tolist()
features = [c for c in features if c not in [TARGET, GROUP]]

X = df[features].fillna(0)
y = df[TARGET]
groups = df[GROUP]

print("Number of features:", len(features))
print("Dataset shape:", X.shape)

Number of features: 29
Dataset shape: (26612, 29)


In [28]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42)),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        class_weight="balanced",
        random_state=42),

    "LightGBM": lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        verbose=-1)}

In [29]:
gkf = GroupKFold(n_splits=5)
results = []
for name, model in models.items():
    precision = []
    recall = []
    f1 = []
    auc = []
    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)
        probabilities = model.predict_proba(X_val)[:, 1]
        precision.append(
            precision_score(y_val, predictions, zero_division=0))
        recall.append(
            recall_score(y_val, predictions, zero_division=0))
        f1.append(
            f1_score(y_val, predictions, zero_division=0))
        auc.append(
            roc_auc_score(y_val, probabilities))

    results.append({
        "Model": name,
        "Precision": np.mean(precision),
        "Recall": np.mean(recall),
        "F1-Score": np.mean(f1),
        "ROC-AUC": np.mean(auc)})

In [30]:
comparison_df = pd.DataFrame(results)
print("MODEL COMPARISON")
display(comparison_df)

MODEL COMPARISON


,Model,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.882129,0.781238,0.828585,0.928638
1,Random Forest,0.741525,0.668865,0.703264,0.817558
2,LightGBM,0.947794,0.979620,0.963431,0.994809


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_model = models["LightGBM"]
best_model.fit(X, y)

importance = permutation_importance(
    best_model,
    X,
    y,
    n_repeats=10,
    random_state=42)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

print("Feature Importance")
display(importance_df)

Feature Importance


,Feature,Importance
15,impressions_last_30d,0.484868
18,impressions_prev_30d,0.373114
21,content_age_days,0.001165
26,engagement_rate,0.000008
0,search_volume,0.000000
9,users_90d,0.000000
6,clicks_90d,0.000000
7,pageviews_90d,0.000000
8,sessions_90d,0.000000
1,competition,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.